# Tools
Models can request to call tools that perform tasks such as fetching data from the database, searching the web, or running code. Tools are pairing of:
    
    1. A schema, including the name of the tool, description, and/or argument definition(often  a   JSON schema)
    2. A function or coroutine to execute.

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='<think>\nOkay, so why do parrots talk? I remember seeing parrots on TV that can mimic human speech, but I\'m not sure why they actually do that. I think it\'s related to their need to communicate, but maybe there\'s more to it. Let me think.\n\nFirst, parrots are social animals. They live in flocks in the wild, so maybe they use vocalizations to interact with each other. But when they mimic human words, is it just an extension of their natural behavior to imitate sounds? I\'ve heard that some parrots can learn words from their owners, but why do they choose certain words over others?\n\nI also recall reading that parrots have a part of their brain called the "vocal learning" region, which allows them to mimic sounds. But how does that relate to their talking ability? Maybe they have a strong auditory memory? Or is it a form of play or bonding with humans?\n\nAnother angle is that talking might be a way for parrots to get attention or rewards. If a parrot says a word 

In [2]:
from langchain_core.tools import retriever
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"

model_with_tools = model.bind_tools([get_weather])

In [3]:
response = model_with_tools.invoke("What's the weather in Boston?")
print(response)
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Boston. I need to use the get_weather function. Let me check the function\'s parameters. The required parameter is location, which should be a string. Since the user mentioned Boston, I should call the function with "Boston" as the location. Make sure to format the tool call correctly within the XML tags.\n', 'tool_calls': [{'id': '6rj9k658y', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 97, 'prompt_tokens': 153, 'total_tokens': 250, 'completion_time': 0.15163115, 'completion_tokens_details': {'reasoning_tokens': 73}, 'prompt_time': 0.006036723, 'prompt_tokens_details': None, 'queue_time': 0.053569226, 'total_time': 0.157667873}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_pr

### Tool execution Loops

In [4]:
# Step 1: Model generated tool calls
messages = [{"role":"user", "content":"What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools to get results
for tool_call in ai_msg.tool_calls:
    #Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass the results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)

The weather in Boston is currently sunny. Let me know if you'd like to check another location! ☀️


In [5]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Boston. I need to use the get_weather function. The function requires a location parameter. Boston is the location here. So I should call get_weather with location set to "Boston". Let me make sure there\'s no typo. Everything looks good. I\'ll format the tool call as specified.\n', 'tool_calls': [{'id': 'cqqb3k9f9', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 93, 'prompt_tokens': 153, 'total_tokens': 246, 'completion_time': 0.148031629, 'completion_tokens_details': {'reasoning_tokens': 69}, 'prompt_time': 0.006650422, 'prompt_tokens_details': None, 'queue_time': 0.048827158, 'total_time': 0.154682051}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_